# Importing libraries

In [ ]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import shap

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df=pd.read_csv("data/data.csv")
df

In [ ]:
df['ClinVar_Significance_Label'].value_counts()

In [ ]:
df['Known_Regulatory_Effect'].value_counts()

In [ ]:
print(df.value_counts(['Known_Regulatory_Effect', 'ClinVar_Significance_Label']))

# Data Exploration, Preprocessing


Note:
Variant_Type: 0=SNV, 1=indel, 2=deletion, 3=insertion, 4=other

Non_Coding_Region: 0=3'UTR, 1=5'UTR, 2=intronic, 3=splice region, 4=upstream, 5=downstream, 6=intergenic, 7=nc transcript, 8=regulatory, 9=other

ClinVar_Significance: -1=missing, 0=benign, 1=likely-benign, 2=uncertain, 3=likely-pathogenic, 4=pathogenic
* using this as label, with just "benign", "likely-benign", "pathogenic","likely-pathogenic"

* Drop unlabelled/uncertain data
* Remove columns with single-valued or null-valued values:

In [ ]:

data = df
data["ClinVar_Significance_Label"].value_counts()
for col in data.columns: # drop homogeneous columns
    if(len(data[col].value_counts())==1):
        data.drop(col, axis=1, inplace=True)
data.drop("H3K27ac_CellLines",axis=1,inplace=True)
data.drop("Known_Regulatory_Effect",axis=1,inplace=True)

see dataframe column info, validate that the remaining data is what we want


In [ ]:
data.info()
for col in data.columns:
    print(f"--------- Value counts for {col}")
    print(data[col].value_counts())
    print(len(data[col].value_counts()))

In [ ]:
display(data)

Alternative: Label Benign samples as 0, Pathogenic as 1? compare models...?

In [ ]:
def add_clinvar_target_label(x):
    if x == "benign" or x == "benign-likely-benign":
        return 0
    elif x == "likely-benign":
        return 1
    elif x == "likely-pathogenic":
        return 2
    elif x == "pathogenic" or x == "pathogenic-likely-pathogenic":
        return 3
    else:
        return np.nan


NOTE: skipping  'Sequence_Alt', 'Sequence_Ref' - change this if needed

In [ ]:
non_categorical_data = ['Position', 'H3K4me3', 'H3K27ac','GTEx_eQTL', 'Training_Split',  'H3K4me3_PeakCount', 'H3K4me3_PeakScore', 'H3K27ac_PeakScore','PhyloP_Score', 'TF_MNT_disrupted', 'TF_MNT_created', 'TF_MAX_disrupted', 'TF_MAX_created', 'TF_ELF1_disrupted', 'TF_CREB1_disrupted', 'TF_CREB1_created', 'TF_TOE1_created', 'TF_ELK1_disrupted', 'TF_OVOL1_created', 'TF_ZFX_disrupted', 'TF_ZFX_created', 'TF_FOS_disrupted', 'TF_FOS_created', 'TF_FOXA1_created', 'TF_JUND_disrupted', 'TF_MSX2_created', 'TF_MYC_disrupted', 'TF_GATA3_disrupted', 'TF_GATA3_created', 'TF_NR2F2_disrupted', 'TF_NR2F2_created', 'TF_E4F1_disrupted', 'TF_E4F1_created', 'TF_E2F8_disrupted', 'TF_E2F8_created', 'TF_FOXM1_created', 'TF_FOSL2_disrupted', 'Max_Disruption_Score', 'Motif_Score_Reliable']

In [ ]:
encoded_columns=[]
nc_region_label = pd.get_dummies(data["Non_Coding_Region_Label"],prefix="nc_region",dtype=int)
encoded_columns.append(nc_region_label)

encode_gene_symbol = pd.get_dummies(data["Gene_Symbol"],prefix="gene_symbol", dtype=int)
encoded_columns.append(encode_gene_symbol)

encode_chromosome = pd.get_dummies(data["Chromosome"],dtype=int, prefix="chromosome")
encoded_columns.append(encode_chromosome)

encode_variant_type = pd.get_dummies(data["Variant_Type_Label"],dtype=int) # this ok?
encoded_columns.append(encode_variant_type)

enc_Top_TF_Bind = pd.get_dummies(data["Top_TF_Binding"],dtype=int, prefix='TF_binding')
encoded_columns.append(enc_Top_TF_Bind)

enc_seq_status = pd.get_dummies(data["Sequence_Status"],dtype=int, prefix="status")
encoded_columns.append(enc_seq_status)

enc_count_dis = pd.get_dummies(data["TF_Count_Disrupted"],dtype=int,prefix="TF_count_disrupt")
encoded_columns.append(enc_count_dis)

enc_tf_count_create = pd.get_dummies(data["TF_Count_Created"],dtype=int,prefix = "TF_Count_Created")
encoded_columns.append(enc_tf_count_create)

enc_tf_most_disrupt = pd.get_dummies(data["Most_Disrupted_TF"],dtype=int, prefix="disrupted_tf")
encoded_columns.append(enc_tf_most_disrupt)

enc_tf_most_create = pd.get_dummies(data["Most_Created_TF"],dtype=int, prefix="created_tf")
encoded_columns.append(enc_tf_most_create)



Put everything into dataframe model will use. Convert dataframe -> np array

In [ ]:
labels_out =  ["uncertain-significance", "not-provided", "conflicting-interpretations-of-pathogenicity"]

In [ ]:
model_data = pd.DataFrame()
model_data = pd.concat(encoded_columns, axis=1)
model_data = pd.concat([model_data, data[non_categorical_data]], axis=1)
model_data["target_label"] = data["ClinVar_Significance_Label"]


unknown_model_data = model_data[
    model_data["target_label"].isin(labels_out)
].copy()
unknown_model_data.drop(["Training_Split", 'target_label'],axis=1,inplace=True)
model_data = model_data[
    ~model_data["target_label"].isin(labels_out)
]
model_data["target_label"] = model_data["target_label"].map(add_clinvar_target_label)

_before = len(model_data)
model_data = model_data.dropna(subset=["target_label"])
if len(model_data) < _before:
    print(f"Dropped {_before - len(model_data)} labeled rows with unmapped ClinVar labels (unexpected values).")


In [ ]:
model_data.info()

In [ ]:
model_data_train = model_data[model_data["Training_Split"] == "Train"]
model_data_train = model_data_train.drop(columns="Training_Split")
model_data_val = model_data[model_data["Training_Split"] == "Validation"]
model_data_val = model_data_val.drop(columns="Training_Split")
model_data_test = model_data[model_data["Training_Split"] == "Test"]
model_data_test = model_data_test.drop(columns="Training_Split")

x_data_train = model_data_train.drop(columns="target_label")
y_data_train = model_data_train["target_label"]
x_data_val = model_data_val.drop(columns="target_label")
y_data_val = model_data_val["target_label"]
x_data_test = model_data_test.drop(columns="target_label")
y_data_test = model_data_test["target_label"]


In [ ]:
x_data_train

In [ ]:
#Verifying that model data and the unknown model data have the same columns

print(len(x_data_train.columns))
print(len(unknown_model_data.columns))
print(x_data_train.columns == unknown_model_data.columns)

Scaling features

In [ ]:
# Scaling is applied inside the Pipeline in the next cell (StandardScaler fit on training
# folds only during tuning; validation selects C and gamma; final refit on train+validation).


In [ ]:
svc_param_grid = {
    "clf__C": [0.1, 1.0, 10.0],
    "clf__gamma": ["scale", "auto"],
}

best_val_score = -np.inf
best_params = None

#tune pipes scales data and tests hyperparameters
tune_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                decision_function_shape="ovr",
                probability=True,
                random_state=42,
            ),
        ),
    ]
)

#The hyperparameter tuning happens here
for params in ParameterGrid(svc_param_grid):
    tune_pipe.set_params(**params)
    tune_pipe.fit(x_data_train, y_data_train)
    val_pred = tune_pipe.predict(x_data_val)
    score = balanced_accuracy_score(y_data_val, val_pred)
    if score > best_val_score:
        best_val_score = score
        best_params = params.copy()

print("Best hyperparameters (by validation balanced accuracy):", best_params)
print("Validation balanced accuracy:", best_val_score)

#joining the training and validation set together to train the best model
x_data_trainval = pd.concat([x_data_train, x_data_val], axis=0)
y_data_trainval = pd.concat([y_data_train, y_data_val], axis=0)

svm_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                decision_function_shape="ovr",
                probability=True,
                random_state=42,
            ),
        ),
    ]
)
svm_pipeline.set_params(**best_params)
svm_pipeline.fit(x_data_trainval, y_data_trainval)

predictions = svm_pipeline.predict(x_data_test)



In [ ]:
print(f"Overall Accuracy: {accuracy_score(y_data_test, predictions)}\n")
print("Balanced accuracy:", balanced_accuracy_score(y_data_test, predictions))
print(classification_report(y_data_test, predictions))
print("--- Classification Report ---")
print(classification_report(y_data_test, predictions))
print("--- Confusion Matrix ---")
cm = confusion_matrix(y_data_test, predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Benign (0)", "Likely Benign (1)", "Likely Patho (2)", "Patho (3)"],
            yticklabels=["Benign (0)", "Likely Benign (1)", "Likely Patho (2)", "Patho (3)"])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('SVM Confusion Matrix')
plt.show()

In [ ]:
_label_map = {
    0: "benign",
    1: "likely_benign",
    2: "likely_pathogenic",
    3: "pathogenic",
}

unknown_pred_class = svm_pipeline.predict(unknown_model_data)
unknown_proba = svm_pipeline.predict_proba(unknown_model_data)

#gets the rows of the data that have the same index as the rows in the unknown_predictions_df
unknown_predictions_df = data.loc[unknown_model_data.index].copy()

unknown_predictions_df["svc_pred_class"] = unknown_pred_class
unknown_predictions_df["predicted_pathogenicity"] = unknown_predictions_df["svc_pred_class"].map(_label_map)

# --- Probabilities (Platt scaling on SVC; approximate but useful for ranking) ---
# Column order of predict_proba follows svm_pipeline.classes_
_class_to_idx = {int(c): j for j, c in enumerate(svm_pipeline.classes_)}
for c in svm_pipeline.classes_:
    unknown_predictions_df[f"svc_proba_class_{int(c)}"] = unknown_proba[:, _class_to_idx[int(c)]]

unknown_predictions_df["svc_proba_pathogenic"] = unknown_predictions_df["svc_proba_class_3"]
unknown_predictions_df["svc_proba_likely_pathogenic"] = unknown_predictions_df["svc_proba_class_2"]
unknown_predictions_df["svc_proba_pathogenic_spectrum"] = (
    unknown_predictions_df["svc_proba_likely_pathogenic"] + unknown_predictions_df["svc_proba_pathogenic"]
)
# How confident the model is in its single top-1 class (any label)
unknown_predictions_df["svc_confidence_top_class"] = unknown_proba.max(axis=1)


# Stores the original index of the data row as its own column in the unknown_predictions df so we can figure out which row maps to where
unknown_predictions_df["original_data_row_index"] = unknown_predictions_df.index


unknown_predictions_df["predicted_pathogenic"] = unknown_predictions_df["svc_pred_class"] == 3

unknown_predictions_df["predicted_pathogenic_spectrum"] = unknown_predictions_df["svc_pred_class"].isin([2, 3])

pathogenic_unknown = unknown_predictions_df[unknown_predictions_df["predicted_pathogenic"]].copy()
display(pathogenic_unknown)


In [ ]:
pathogenic_unknown.to_csv("pathogenic_unknowns.csv")

## SHAP feature influence (global)

Model-agnostic **permutation SHAP** approximates how much each feature pushes `predict_proba` away from the baseline (average over the background sample). The pipeline’s scaler runs inside `predict_proba`, so attributions match what the SVC actually sees.

Reduce `N_BG` / `N_EXPLAIN` / `max_evals` if this cell is slow.


In [ ]:
# --- SHAP: which features matter for the fitted pipeline? ---
# Uses predict_proba (requires probability=True on SVC — set in the training cell above).

N_BG = min(80, len(x_data_trainval))
N_EXPLAIN = min(40, len(x_data_test))

X_shap_bg = shap.sample(x_data_trainval, N_BG, random_state=42)
X_shap_explain = x_data_test.sample(N_EXPLAIN, random_state=43)

masker = shap.maskers.Independent(X_shap_bg)
explainer = shap.PermutationExplainer(
    svm_pipeline.predict_proba,
    masker,
    feature_names=list(x_data_trainval.columns),
)

# Lower max_evals for a faster approximate run; must be >= 2 * n_features + 1 for PermutationExplainer.
_min_evals = 2 * X_shap_explain.shape[1] + 1
shap_explanation = explainer(
    X_shap_explain,
    max_evals=max(_min_evals, min(800, 12 * X_shap_explain.shape[1])),
)

vals = np.asarray(shap_explanation.values)
if vals.ndim != 3:
    raise ValueError(f"Expected SHAP values shaped (samples, features, classes); got {vals.shape}")

feature_names = list(x_data_trainval.columns)
class_names = [
    "benign (0)",
    "likely_benign (1)",
    "likely_pathogenic (2)",
    "pathogenic (3)",
]


def plot_top_features_for_class(class_idx: int, top_k: int = 20) -> None:
    mean_abs = np.abs(vals[:, :, class_idx]).mean(axis=0)
    top = pd.Series(mean_abs, index=feature_names).sort_values(ascending=False).head(top_k)
    fig, ax = plt.subplots(figsize=(10, max(4, top_k * 0.25)))
    top.iloc[::-1].plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("mean |SHAP value| (permutation)")
    ax.set_title(f"Top features — influence on P(class | x): {class_names[class_idx]}")
    plt.tight_layout()
    plt.show()


# Pathogenic (3) and likely-pathogenic (2) are usually the priority readouts
plot_top_features_for_class(3, top_k=20)
plot_top_features_for_class(2, top_k=20)

# Overall influence across all class outputs (single ranking)
mean_abs_all_outputs = np.abs(vals).mean(axis=(0, 2))
top_global = (
    pd.Series(mean_abs_all_outputs, index=feature_names)
    .sort_values(ascending=False)
    .head(25)
)
fig, ax = plt.subplots(figsize=(10, 7))
top_global.iloc[::-1].plot.barh(ax=ax, color="darkslategray")
ax.set_xlabel("mean |SHAP value|, averaged over samples and classes")
ax.set_title("Top features — global influence (all classes combined)")
plt.tight_layout()
plt.show()
